# 02 · Analyst Agent

**Role in the pipeline:** quantify *how much* risk each flagged account carries and *why*.

- **CLV model** — XGBoost regression, next-12-month value
- **Churn model** — XGBoost classifier, probability of churn in 6 months
- **Explanations** — TreeSHAP contributions per account (native XGBoost, no `shap` package needed)
- **Next best action** — resolved from the editable YAML playbook

Output per account: `{clv_score, churn_probability, revenue_at_risk, risk_level, top_factors}`

In [ ]:
# --- Setup: point the notebook at the project's `copilot` package ---------
# Must run BEFORE any `copilot` import: settings are read once from env vars.
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "copilot").exists():          # works from notebooks/ or project root
    if ROOT == ROOT.parent:
        raise RuntimeError("Open the cs_ai_copilot_dss folder in VS Code and run from there.")
    ROOT = ROOT.parent
os.chdir(ROOT)                                            # playbook path is relative to project root
sys.path.insert(0, str(ROOT / "src"))

DEV = ROOT / "notebooks" / "_dev"                         # local, git-ignorable sandbox
DEV.mkdir(parents=True, exist_ok=True)
os.environ["DATABASE_URL"] = f"sqlite:///{(DEV / 'copilot_dev.db').as_posix()}"
os.environ["MODEL_DIR"] = str(DEV / "models")
os.environ.pop("CHROMA_HOST", None)                       # notebooks use the in-memory RAG store

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Project root:", ROOT)
print("Dev database:", os.environ["DATABASE_URL"])

## 1 · Load data and the Monitoring hand-off

In [ ]:
from copilot.data import load_gold
from copilot.agents.monitoring import MonitoringAgent

gold = load_gold()
handoff_file = DEV / "flagged.json"
if handoff_file.exists():
    flagged = json.loads(handoff_file.read_text())["flagged"]
    print(f"Loaded {len(flagged)} flagged accounts from 01_monitoring_agent")
else:                                                    # lets this notebook run standalone
    flagged = MonitoringAgent().run(gold).flagged
    print(f"No hand-off file — ran Monitoring inline: {len(flagged)} flagged")

## 2 · Train and evaluate
Models train on **all** accounts (75/25 split), then score only the flagged ones. Metrics are on the held-out 25%.

In [ ]:
from copilot.agents.analyst import AnalystAgent
from copilot.config import get_settings

s = get_settings()
analyst = AnalystAgent(s.model_dir, risk_high=s.risk_high, risk_medium=s.risk_medium)
metrics = analyst.train(gold)
pd.Series(metrics).round(3).to_frame("held-out test")

**Reading the metrics**
- `churn_roc_auc` — how well the model ranks churners above non-churners (0.5 = coin flip).
- `churn_precision@high` — of accounts labelled *high risk*, how many really churned. This drives how much reviewer time is well spent.
- `churn_recall@high` — of real churners, how many reached *high risk*.
- `churn_base_rate` — compare precision against this: precision must beat it to add value.

### Choose the high-risk threshold
Precision/recall trade-off on the test split. Production reads `RISK_HIGH` from `.env`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from copilot.agents.analyst import encode

X = encode(gold); y = gold.churned_next_6m
_, X_te, _, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p_te = analyst.churn_model.predict_proba(X_te)[:, 1]

thr = [round(t, 2) for t in pd.Series(range(20, 90, 5)) / 100]
curve = pd.DataFrame({"threshold": thr,
                      "precision": [precision_score(y_te, p_te >= t, zero_division=0) for t in thr],
                      "recall": [recall_score(y_te, p_te >= t) for t in thr],
                      "% flagged high": [(p_te >= t).mean() for t in thr]})
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(curve.threshold, curve.precision, marker="o", label="precision")
ax.plot(curve.threshold, curve.recall, marker="o", label="recall")
ax.axvline(s.risk_high, ls="--", c="grey"); ax.text(s.risk_high + 0.01, 0.05, "RISK_HIGH", color="grey")
ax.axhline(y_te.mean(), ls=":", c="#c0703b"); ax.text(0.2, y_te.mean() + 0.02, "base rate", color="#c0703b")
ax.set_xlabel("churn-probability threshold"); ax.set_ylim(0, 1.02); ax.legend()
ax.set_title("High-risk threshold trade-off")
plt.tight_layout(); plt.show()
curve.round(2)

## 3 · Global explanation — what drives churn risk overall?
Mean absolute TreeSHAP contribution across all accounts. `supplier_origin` one-hot columns are folded into one factor.

In [ ]:
import numpy as np, xgboost as xgb
from copilot.agents.analyst import FRIENDLY

contrib = analyst.churn_model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)[:, :-1]
cdf = pd.DataFrame(contrib, columns=X.columns)
origin_cols = [c for c in cdf if c.startswith("supplier_origin=")]
cdf["supplier_origin"] = cdf[origin_cols].sum(axis=1)
importance = cdf.drop(columns=origin_cols).abs().mean().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.8))
ax.barh([FRIENDLY.get(c, c) for c in importance.index], importance.values, color="#3b6ea8")
ax.set_xlabel("mean |SHAP| (log-odds)"); ax.set_title("Churn drivers across the customer base")
plt.tight_layout(); plt.show()

> Check: *Average order value* should **not** dominate. If it does, the model is learning account size rather than behaviour — the log transform in `data.py` is what fixes this.

## 4 · Score the flagged accounts

In [ ]:
assessments = analyst.assess(gold, list(flagged))
assessments.sort(key=lambda a: -a.revenue_at_risk)       # pilot rule: prioritise by € at risk

scored = pd.DataFrame([{
    "account_id": a.account_id, "risk_level": a.risk_level,
    "churn_prob": a.churn_probability, "clv_€": a.clv_score, "revenue_at_risk_€": a.revenue_at_risk,
    "top_driver": next((f["label"] for f in a.top_factors if f["direction"] == "increases risk"), "—"),
} for a in assessments])
print(scored.risk_level.value_counts().to_string())
scored.head(15).round(2)

## 5 · Explain one account
This is what the reviewer sees next to the draft. Change `ACCOUNT` to inspect any flagged account.

In [ ]:
ACCOUNT = assessments[0].account_id                      # highest revenue at risk

a = next(x for x in assessments if x.account_id == ACCOUNT)
f = pd.DataFrame(a.top_factors)
fig, ax = plt.subplots(figsize=(7, 3))
colors = ["#c0503b" if v > 0 else "#3b8a5a" for v in f.shap_log_odds]
ax.barh([f"{l}  ({v})" for l, v in zip(f.label, f.value)][::-1], f.shap_log_odds[::-1], color=colors[::-1])
ax.axvline(0, c="black", lw=0.8)
ax.set_xlabel("← reduces risk      SHAP (log-odds)      increases risk →")
ax.set_title(f"{ACCOUNT}: churn {a.churn_probability:.0%} · CLV €{a.clv_score:,.0f} · {a.risk_level}")
plt.tight_layout(); plt.show()
print("Monitoring signals:", "; ".join(flagged[ACCOUNT]))

## 6 · Next best action from the playbook
Edit `playbook/nba_playbook.yaml` and re-run this cell — no retraining needed.

In [ ]:
from copilot.agents.nba import recommend_next_best_action

segments = gold.set_index("account_id").segment
nba = pd.DataFrame([{"account_id": a.account_id, "risk_level": a.risk_level,
                     "segment": segments[a.account_id],
                     **recommend_next_best_action(a.risk_level, segments[a.account_id],
                                                  a.top_factors, s.playbook_path).__dict__}
                    for a in assessments])
display(nba.action.value_counts().to_frame("accounts"))
nba[["account_id", "risk_level", "segment", "rule_id", "action", "owner"]].head(10)

## 7 · Hand-off: persist scores for the Human Review notebook
Writes to the same `risk_scores` table the API reads in Docker.

In [ ]:
from datetime import datetime, timezone
from copilot.db import Account, RiskScore, SessionLocal, init_db
from copilot.data import FEATURES

init_db()
RUN_ID = "nb-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
with SessionLocal() as session:
    for row in gold.itertuples():
        session.merge(Account(account_id=row.account_id, name=row.name, segment=row.segment,
                              supplier_origin=row.supplier_origin,
                              features={c: (getattr(row, c).item() if hasattr(getattr(row, c), "item")
                                            else getattr(row, c)) for c in FEATURES}))
    for a in assessments:
        session.add(RiskScore(run_id=RUN_ID, account_id=a.account_id, clv_score=a.clv_score,
                              churn_probability=a.churn_probability, risk_level=a.risk_level,
                              top_factors=a.top_factors, monitor_signals=flagged[a.account_id]))
    session.commit()
(DEV / "last_run.txt").write_text(RUN_ID)
print(f"Saved {len(assessments)} assessments under run_id {RUN_ID}")
print("Next: open 03_human_review_agent.ipynb")